<a href="https://colab.research.google.com/github/shreyansh8h/hello-world/blob/master/May16_Arul_multi_agent_orchestration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 - Multi-Agent Orchestration with LangChain and LLM Reasoning

## What you will learn

By the end, you should be able to:
- Model multiple specialized agents.
- Use LangChain chains for agent roles.
- Coordinate agents through an orchestrator.
- Add review and improvement loops.

This notebook uses LangChain with `ChatOpenAI` for real LLM reasoning. Set `OPENAI_API_KEY` before running the LLM cells.

In [3]:
!pip -q install -U "langchain-huggingface" "transformers>=4.45,<4.58" "tokenizers<0.22" accelerate bitsandbytes "langchain-openai"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 13.4 MB/s eta 0:00:00


In [4]:
import os
import re
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI


def make_llm(temperature=0):
    """Create a real OpenAI-backed chat model for the notebook examples."""
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError(
            "OPENAI_API_KEY is not set. Set it before running LLM cells so the notebook uses a real model."
        )
    return ChatOpenAI(
        model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
        temperature=temperature,
    )


def print_box(title, text):
    print(f"\n{title}\n" + "-" * len(title))
    print(text)


print("LangChain setup ready.")
print("LLM provider: OpenAI via ChatOpenAI")
print("Model:", os.getenv("OPENAI_MODEL", "gpt-4o-mini"))
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))

LangChain setup ready.
LLM provider: OpenAI via ChatOpenAI
Model: mistralai/Mistral-7B-Instruct-v0.3
HF_TOKEN configured: False


## 1. Create Role-Specific LLM Agents

In [ ]:
researcher_llm = make_llm()
analyst_llm = make_llm()
writer_llm = make_llm()
reviewer_llm = make_llm()


def role_chain(role, llm):
    prompt = ChatPromptTemplate.from_messages([
        ("system", f"You are the {role} agent. Be concise and practical."),
        ("human", "{task}"),
    ])
    return prompt | llm | StrOutputParser()


researcher = role_chain("researcher", researcher_llm)
analyst = role_chain("analyst", analyst_llm)
writer = role_chain("writer", writer_llm)
reviewer = role_chain("reviewer", reviewer_llm)

## 2. Sequential Orchestration

In [ ]:
def sequential_orchestrator(topic):
    research = researcher.invoke({"task": f"Collect key facts about {topic}."})
    analysis = analyst.invoke({"task": f"Analyze these findings: {research}"})
    draft = writer.invoke({"task": f"Write a student-friendly summary using this analysis: {analysis}"})
    review = reviewer.invoke({"task": f"Review this draft: {draft}"})
    return {
        "topic": topic,
        "research": research,
        "analysis": analysis,
        "draft": draft,
        "review": review,
    }


result = sequential_orchestrator("agent architectures")
for key, value in result.items():
    print_box(key, value)

## 3. Parallel Specialist Agents

In [ ]:
reactive_llm = make_llm()
planning_llm = make_llm()
coordination_llm = make_llm()

reactive_specialist = role_chain("reactive architecture specialist", reactive_llm)
planning_specialist = role_chain("planning architecture specialist", planning_llm)
coordination_specialist = role_chain("multi-agent coordination specialist", coordination_llm)

specialist_panel = RunnableParallel(
    reactive=reactive_specialist,
    planning=planning_specialist,
    coordination=coordination_specialist,
)

panel_result = specialist_panel.invoke({
    "task": "Give one practical use case and one limitation for your architecture."
})

print(panel_result)

## 4. Aggregator Agent

In [ ]:
aggregator_llm = make_llm()

aggregator_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an aggregator agent. Combine specialist views into one concise recommendation."),
    ("human", "Specialist outputs: {panel_result}"),
])

aggregator = aggregator_prompt | aggregator_llm | StrOutputParser()
recommendation = aggregator.invoke({"panel_result": panel_result})
print(recommendation)

## 5. Review Loop

In [ ]:
critic_llm = make_llm()

critic_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a critic agent. Return 'approved:' or 'revise:' with one reason."),
    ("human", "{draft}"),
])

critic = critic_prompt | critic_llm | StrOutputParser()

first_review = critic.invoke({"draft": recommendation})
revised = recommendation + " Workflow-oriented systems are useful when the process is known and must be auditable."
second_review = critic.invoke({"draft": revised})

print_box("First review", first_review)
print_box("Revised recommendation", revised)
print_box("Second review", second_review)

## Exercise

Add a `risk_agent` that identifies one coordination risk in the recommendation.

Questions to answer:
- What happens when two agents disagree?
- Should the orchestrator always obey the reviewer?
- How can message logs help debug multi-agent systems?

In [ ]:
risk_llm = make_llm()

risk_agent = role_chain("risk agent", risk_llm)
risk = risk_agent.invoke({"task": f"Identify one risk in this plan: {revised}"})
print(risk)

## Key Takeaway

Multi-agent orchestration uses specialization plus coordination. LangChain makes role chains, parallel execution, aggregation, and review loops straightforward to demonstrate.